In [1]:
import tensorflow as tf
import keras
import numpy as np
import sys

sys.path.append("../")

In [2]:
DATA_DIR = "../mu3e_trigger_data"
SIGNAL_PIXEL_FILE = f"{DATA_DIR}/run42_sig_pixel_128.npy"
BACKGROUND_PIXEL_FILE = f"{DATA_DIR}/run42_bg_pixel_128.npy"
SIGNAL_DATA_FILE = f"{DATA_DIR}/run42_sig_mppc_128.npy"
BACKGROUND_DATA_FILE = f"{DATA_DIR}/run42_bg_mppc_128.npy"

bg_pixel_spacetime = np.load(BACKGROUND_PIXEL_FILE)
sig_pixel_spacetime = np.load(SIGNAL_PIXEL_FILE)
bg_mppc_spacetime = np.load(BACKGROUND_DATA_FILE)
sig_mppc_spacetime = np.load(SIGNAL_DATA_FILE)

input_seq_len = bg_pixel_spacetime.shape[1]
input_dim = bg_pixel_spacetime.shape[2] - 1  # Exclude timestamp

In [3]:
pixel_input = keras.Input(shape=(128, 4), name="pixel_input")
mppc_input = keras.Input(shape=(128, 3), name="mppc_input")
feature_dim = 8
latent_dim = 16

In [4]:
from src.model.components import (
    point_transformer,
    SelfAttentionStack,
    MultiHeadAttentionBlock,
    MLP,
    GenerateMask,
    PoolingAttentionBlock,
    MultiHeadAttentionStack,
)


pixel_mask = GenerateMask(-1, name="pixel_mask")(pixel_input)
mppc_mask = GenerateMask(-1, name="mppc_mask")(mppc_input)

pixel_embedding = MLP(output_dim=feature_dim, name="pixel_embedding")(pixel_input)

mppc_embedding = MLP(output_dim=feature_dim, name="mppc_embedding")(mppc_input)

pixel_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_attention",
)(pixel_embedding, pixel_mask)

mppc_attention = SelfAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="mppc_attention",
)(mppc_embedding, mppc_mask)

mppc_attend_pixel = MultiHeadAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="mppc_attend_pixel",
)(query = mppc_attention, value =  pixel_attention, query_mask=mppc_mask, value_mask=pixel_mask)

pixel_attend_mppc = MultiHeadAttentionStack(
    num_heads=4,
    key_dim=feature_dim,
    stack_size=3,
    name="pixel_attend_mppc",
)(query = pixel_attention, value = mppc_attention, query_mask=pixel_mask, value_mask=mppc_mask)

pixel_attentions_pool = PoolingAttentionBlock(
    name="pixel_attentions_pool",
    key_dim=feature_dim,
    num_seeds=1,
    num_heads=4,
)(mppc_attend_pixel, pixel_mask)

mppc_attentions_pool = PoolingAttentionBlock(
    name="mppc_attentions_pool",
    key_dim=feature_dim,
    num_seeds=1,
    num_heads=4,
)(pixel_attend_mppc, mppc_mask)

latent_space = keras.layers.Concatenate(name="latent_space")(
    [
        keras.layers.Flatten()(pixel_attentions_pool),
        keras.layers.Flatten()(mppc_attentions_pool),
    ]
)

output = MLP(num_layers=4, output_dim=1, activation="sigmoid", name="output")(
    latent_space
)

model = keras.Model(
    inputs=[pixel_input, mppc_input],
    outputs=output,
    name="ClassificationModel",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.BinaryCrossentropy(),
    metrics=[keras.metrics.BinaryAccuracy()],
)

In [5]:
model.summary()

Model: "ClassificationModel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ mppc_input          │ (None, 128, 3)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_input         │ (None, 128, 4)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_embedding      │ (None, 128, 8)    │        178 │ mppc_input[0][0]  │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_mask           │ (None, 128, 1)    │          0 │ mppc_input[0][0]  │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_embedding     │ (None, 128, 8)    │        234 │ pixel_input[0][0] │
│ (MLP)               │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_mask          │ (None, 128, 1)    │          0 │ pixel_input[0][0] │
│ (GenerateMask)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_attention      │ (None, 128, 8)    │      4,320 │ mppc_embedding[0… │
│ (SelfAttentionStac… │                   │            │ mppc_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_attention     │ (None, 128, 8)    │      4,320 │ pixel_embedding[… │
│ (SelfAttentionStac… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_attend_pixel   │ (None, 128, 8)    │      4,320 │ mppc_attention[0… │
│ (MultiHeadAttentio… │                   │            │ mppc_mask[0][0],  │
│                     │                   │            │ pixel_attention[… │
│                     │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_attend_mppc   │ (None, 128, 8)    │      4,320 │ pixel_attention[… │
│ (MultiHeadAttentio… │                   │            │ pixel_mask[0][0], │
│                     │                   │            │ mppc_attention[0… │
│                     │                   │            │ mppc_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pixel_attentions_p… │ (None, 1, 8)      │      1,448 │ mppc_attend_pixe… │
│ (PoolingAttentionB… │                   │            │ pixel_mask[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ mppc_attentions_po… │ (None, 1, 8)      │      1,448 │ pixel_attend_mpp… │
│ (PoolingAttentionB… │                   │            │ mppc_mask[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 8)         │          0 │ pixel_attentions… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 8)         │          0 │ mppc_attentions_… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ latent_space        │ (None, 16)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (MLP)        │ (None, 1)         │        370 │ latent_space[0][… │
└─────────────────────┴───────────────────┴────────────┴─────────────────

 Total params: 21,127 (82.53 KB)

 Trainable params: 21,127 (82.53 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
from sklearn.model_selection import train_test_split

(
    X_pixel_train,
    X_pixel_test,
    X_mppc_train,
    X_mppc_test,
    y_train,
    y_test,
) = train_test_split(
    np.concatenate([bg_pixel_spacetime[:, :, :], sig_pixel_spacetime[:, :, :]], axis=0),
    np.concatenate([bg_mppc_spacetime[:, :, :], sig_mppc_spacetime[:, :, :]], axis=0),
    np.concatenate(
        [np.zeros(len(bg_pixel_spacetime)), np.ones(len(sig_pixel_spacetime))]
    ),
    test_size=0.2,
    random_state=42,
)

In [7]:
sample_weights = np.ones_like(y_train)
seq_length = (X_pixel_train != -1).all(axis=-1).sum(axis=-1) + (X_mppc_train != -1).all(
    axis=-1
).sum(axis=-1)

for true_label in np.unique(y_train):
    for length in np.unique(seq_length):
        sample_weights[(seq_length == length) & (true_label == y_train)] = 1 /( np.sum((seq_length == length) & (true_label == y_train)) +1e-6)

sample_weights /= np.mean(sample_weights)

In [ ]:
model.fit(
    x=[X_pixel_train, X_mppc_train],
    y=y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=512,
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=4, restore_best_weights=True
        )
    ],
    sample_weight=sample_weights,
)

Epoch 1/10
 21/174 ━━━━━━━━━━━━━━━━━━━━ 7:15 3s/step - binary_accuracy: 0.4351 - loss: 0.6860

In [ ]:
predictions = model.predict([X_pixel_test, X_mppc_test])
seq_length = (X_pixel_test != -1).all(axis=-1).sum(axis=-1) + (X_mppc_test != -1).all(
    axis=-1
).sum(axis=-1)
mppc_length = (X_mppc_test != -1).all(axis=-1).sum(axis=-1)
pixel_length = (X_pixel_test != -1).all(axis=-1).sum(axis=-1)

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc

fpr, tpr, thresholds = roc_curve(y_test, predictions)
fpr_seq_length, tpr_seq_length, thresholds_seq_length = roc_curve(y_test, seq_length)
roc_auc_seq_length = auc(fpr_seq_length, tpr_seq_length)
roc_auc = auc(fpr, tpr)
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color="blue", label="ROC curve (area = {:.2f})".format(roc_auc))
plt.plot(
    fpr_seq_length,
    tpr_seq_length,
    color="green",
    label="ROC curve with number of hits (area = {:.2f})".format(roc_auc_seq_length),
)
plt.title("Receiver Operating Characteristic")
plt.grid()
plt.plot([0, 1], [0, 1], color="red", linestyle="--")
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(loc="lower right")
plt.savefig("roc_curve.png")

In [ ]:
np.corrcoef(predictions.flatten(), seq_length.flatten())

In [ ]:
np.corrcoef(predictions.flatten(), mppc_length)

In [ ]:
np.corrcoef(predictions.flatten(), pixel_length)

In [ ]:
plt.scatter(seq_length.flatten(), predictions.flatten(), alpha=0.5)